## 1. Setup e imports

In [4]:
import sys
import pandas as pd
from dotenv import load_dotenv

# Raiz del proyecto (ejecutar desde notebooks/)
sys.path.append("../../")
load_dotenv("../../.env")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

from src.config.settings import COUNTRY
from src.core.scraper.app import ScrapingUtils
from src.core.scraper.url_filters import filter_product_urls
from src.core.price_tracking import run_price_tracking
from src.utils.clean_model_name import clean_model_name
from src.utils.replace_model_name import map_model_name, load_mapping_file
from src.core.italika_pipeline import build_price_comparison
from src.sources.sheets.reader import GoogleSheetReader

gsheets = GoogleSheetReader()
fr_scraper = ScrapingUtils()

## 2. Configuracion

In [ ]:
BRAND_NAME   = "Italika"
BRAND_URL    = "https://www.italika.mx/"
YEAR_SCRAPED = 2026

## Lectura de base de inventario

Lectura desde archivo local

In [ ]:
# CSV_PATH = rf"C:\Users\JTRUJILLO\Desktop\utiles\Reportes\historical_data\src\data\prices\{COUNTRY}-prices.csv"
# df_inventory = pd.read_csv(CSV_PATH)

Lectura desde archivo en google sheets

In [5]:
google_sheet_info = {
    "sheet_name": '[MKP - MX] Data',
    "worksheet": f'price_data',
}
df_inventory = gsheets.read_sheet(google_sheet_info)

## 3. Cargar inventario

In [ ]:
df_inventory = df_inventory[df_inventory["brand"] == BRAND_NAME]
df_inventory = df_inventory[df_inventory["status"].isin(["available", "no_stock"])] # Modelos disponibles en marketplace
df_inventory

,date,code,brand,model,year,status,price_base,discount_amount,price_net
125,06/04/2026,MX2589-italika-tc-300,Italika,TC 300,2025,available,47999.0,0.0,47999.0
192,06/04/2026,MX2987-italika-ws-175-sport-grafito,Italika,WS 175 Sport Grafito,2026,no_stock,27999.0,0.0,27999.0
217,06/04/2026,MX3395-italika-125z-gps,Italika,125Z GPS,2026,available,27999.0,0.0,27999.0
218,06/04/2026,MX1306-italika-dm-250,Italika,DM 250,2026,no_stock,33999.0,0.0,33999.0
244,06/04/2026,MX2769-italika-at-110-rt,Italika,AT 110 RT,2026,no_stock,18999.0,0.0,18999.0
...,...,...,...,...,...,...,...,...,...
1582,06/04/2026,MX2507-italika-280z,Italika,280Z,2026,no_stock,42999.0,0.0,42999.0
1610,06/04/2026,MX3410-italika-modena-175-gps,Italika,Modena 175 GPS,2026,available,32999.0,0.0,32999.0
1613,06/04/2026,MX3454-italika-firebird-300-gps,Italika,Firebird 300 GPS,2026,no_stock,43999.0,0.0,43999.0
1699,06/04/2026,MX3408-italika-ft-150-gts-gps,Italika,FT 150 GTS GPS,2026,available,24999.0,0.0,24999.0


## 4. Obtener URLs de cada modelo

In [4]:
# Descubrimiento dinamico de URLs desde el sitio
urls = fr_scraper.get_all_urls_from_website(BRAND_URL)
valid_urls = filter_product_urls(urls, exclude_terms=["morbidelli"])
print(f"URLs validas encontradas: {len(valid_urls)}")

URLs validas encontradas: 62


In [5]:
# Lista fija para pruebas puntuales (comentar/descomentar segun necesidad)
italika_urls = [
    "https://www.italika.mx/motocicleta-chopper-italika-rc200-gris-34005722/p",
    "https://www.italika.mx/motoneta-italika-vitalia-150-beige-con-gps-34006773/p",
    "https://www.italika.mx/motocicleta-italika-dm300-dorado-con-gps-34006890/p",
    "https://www.italika.mx/motocicleta-deportiva-italika-vort-x-250-roja-con-negro-34005120/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft125-ts-roja-con-negro-34005164/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft150-ts-negro-con-verde-34005332/p",
    "https://www.italika.mx/motoneta-italika-d150-blanco-con-azul-34006669/p",
    "https://www.italika.mx/cuatrimoto-italika-atv-190-gris-34006332/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-at135x-verde-34006711/p",
    "https://www.italika.mx/motocicleta-urbana-italika-280z-gris-34006326/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-at125-grafito-34006712/p",
    "https://www.italika.mx/cuatrimoto-italika-atv200-naranja-34006718/p",
    "https://www.italika.mx/motocicleta-italika-150z-negra-con-azul-con-gps-34007056/p",
    "https://www.italika.mx/motocicleta-chopper-italika-tc-250-gris-34006691/p",
    "https://www.italika.mx/motoneta-italika-bit-150-blanca-con-negro-34006931/p",
    "https://www.italika.mx/motocicleta-doble-proposito-italika-dm200-con-gps-azul-34006255-1300703478/p",
    "https://www.italika.mx/cuatrimoto-italika-atv300-azul-34006717/p",
    "https://www.italika.mx/motoneta-italika-d125-azul-con-negro-34006289/p",
    "https://www.italika.mx/motoneta-italika-ws175-sport-grafito-34006561/p",
    "https://www.italika.mx/motoneta-italika-ws150-sport-marino-34006551/p",
    "https://www.italika.mx/motocicleta-italika-firebird-300-negra-con-gps-34007122/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft150-roja-34006714/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft200-negra-34006715/p",
    "https://www.italika.mx/motocicleta-doble-proposito-italika-dm150-con-gps-blanca-34006888/p",
    "https://www.italika.mx/motoneta-italika-d150-lt-negra-34005388/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-at110-rt-azul-34006720/p",
    "https://www.italika.mx/motocicleta-deportiva-italika-rt-250-con-gps-roja-34007114/p",
    "https://www.italika.mx/motocicleta-chopper-italika-tc-300-negra-34006362/p",
    "https://www.italika.mx/motocicleta-italika-200z-azul-con-gps-34006885/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft125-negra-34005157/p",
    "https://www.italika.mx/motocicleta-italika-250z-carbono-negra-con-gps-34007058/p",
    "https://www.italika.mx/motocicleta-cafe-racer-italika-sptfire-250-plata-34005271/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft200-gts-gris-con-gps-34007052/p",
    "https://www.italika.mx/motocicleta-deportiva-italika-vort-x-300-roja-con-negro-34004912/p",
    "https://www.italika.mx/motocicleta-deportiva-italika-vort-x-300r-azul-con-gris-34006484/p",
    "https://www.italika.mx/motoneta-italika-modena-175-con-gps-azul-34006716/p",
    "https://www.italika.mx/motocicleta-chopper-italika-rc250-negra-34005813/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-at130-rt-blanca-34006713/p",
    "https://www.italika.mx/motoneta-italika-ds150-negra-34006307/p",
    "https://www.italika.mx/motocicleta-italika-ft150-gts-gris-con-amarillo-con-gps-34007053/p",
    "https://www.italika.mx/motocicleta-de-trabajo-italika-ft250-gts-con-gps-azul-con-negro-34006392/p",
    "https://www.italika.mx/motocicleta-urbana-italika-280z-con-gps-gris-34006887/p",
    "https://www.italika.mx/motocicleta-urbana-italika-125z-gris-con-gps-34006883/p",
]

## 5. Ejecutar price tracking

In [6]:
rows = run_price_tracking(
    urls=italika_urls,
    brand_name=BRAND_NAME,
)

## 6. Limpiar y mapear modelos

In [7]:
df_scraped = pd.DataFrame(rows)
df_scraped["year_scraped"] = YEAR_SCRAPED

df_scraped["model_name_clean"] = df_scraped["model_name"].apply(clean_model_name)

mapeo = load_mapping_file(COUNTRY, BRAND_NAME)
df_scraped["model_mapped"] = df_scraped["model_name_clean"].apply(lambda x: map_model_name(x, mapeo))
df_scraped["model_mapped"] = df_scraped["model_mapped"].str.lower()

df_scraped

Archivo de mapeo cargado correctamente: c:\Users\JTRUJILLO\Desktop\utiles\Proyectos\tracking_prices_changes\notebooks\test\..\..\src/data/json/replace_name/MX/italika_mapeo_nombres.json


,brand_name,model_name,url,price,price_type,currency,captured_at,change_status,previous_scrape_at,visibility,year_scraped,model_name_clean,model_mapped
0,Italika,motocicleta chopper italika rc200 gris,https://www.italika.mx/motocicleta-chopper-italika-rc200-gris-34005722/p,"27,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:05.848+00:00,visible,2026,rc200,rc200
1,Italika,motoneta italika vitalia 150 beige con gps,https://www.italika.mx/motoneta-italika-vitalia-150-beige-con-gps-34006773/p,"31,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:05.51+00:00,visible,2026,vitalia 150 gps,vitalia 150 gps
2,Italika,motocicleta italika dm300 dorado con gps,https://www.italika.mx/motocicleta-italika-dm300-dorado-con-gps-34006890/p,"42,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:06.181+00:00,visible,2026,dm300 gps,dm 300 gps
3,Italika,motocicleta deportiva italika vort x 250 roja con negro,https://www.italika.mx/motocicleta-deportiva-italika-vort-x-250-roja-con-negro-34005120/p,"49,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,changed,2026-03-26T20:26:06.119+00:00,visible,2026,vort x 250,vort x 250
4,Italika,motocicleta de trabajo italika ft125 ts roja con negro,https://www.italika.mx/motocicleta-de-trabajo-italika-ft125-ts-roja-con-negro-34005164/p,"18,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:05.777+00:00,visible,2026,ft125 ts,ft 125 ts
5,Italika,motocicleta de trabajo italika ft150 ts negro con verde,https://www.italika.mx/motocicleta-de-trabajo-italika-ft150-ts-negro-con-verde-34005332/p,"21,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:15.556+00:00,visible,2026,ft150 ts,ft 150 ts
6,Italika,motoneta italika d150 blanco con azul,https://www.italika.mx/motoneta-italika-d150-blanco-con-azul-34006669/p,"22,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:16.38+00:00,visible,2026,d150,d150
7,Italika,cuatrimoto italika atv 190 gris,https://www.italika.mx/cuatrimoto-italika-atv-190-gris-34006332/p,"49,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:16.683+00:00,visible,2026,atv 190,atv 190
8,Italika,motocicleta de trabajo italika at135x verde,https://www.italika.mx/motocicleta-de-trabajo-italika-at135x-verde-34006711/p,"24,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:15.546+00:00,visible,2026,at135x,at135x
9,Italika,motocicleta urbana italika 280z gris,https://www.italika.mx/motocicleta-urbana-italika-280z-gris-34006326/p,"41,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:15.724+00:00,visible,2026,280z,280z


## 7. Merge con inventario (inspeccion)

In [8]:
df_merged = pd.merge(
    df_scraped,
    df_inventory.assign(model_lower=df_inventory["model"].str.lower()),
    left_on="model_mapped",
    right_on="model_lower",
    how="left",
    indicator=True,
)
df_merged

,brand_name,model_name,url,price,price_type,currency,captured_at,change_status,previous_scrape_at,visibility,year_scraped,model_name_clean,model_mapped,date,code,brand,model,year,status,price_base,discount_amount,price_net,model_lower,_merge
0,Italika,motocicleta chopper italika rc200 gris,https://www.italika.mx/motocicleta-chopper-italika-rc200-gris-34005722/p,"27,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:05.848+00:00,visible,2026,rc200,rc200,30/03/2026,MX1314-italika-rc200,Italika,RC200,2025.0,available,27999.0,0.0,27999.0,rc200,both
1,Italika,motocicleta chopper italika rc200 gris,https://www.italika.mx/motocicleta-chopper-italika-rc200-gris-34005722/p,"27,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:05.848+00:00,visible,2026,rc200,rc200,30/03/2026,MX1314-italika-rc200,Italika,RC200,2026.0,no_stock,28999.0,0.0,28999.0,rc200,both
2,Italika,motoneta italika vitalia 150 beige con gps,https://www.italika.mx/motoneta-italika-vitalia-150-beige-con-gps-34006773/p,"31,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:05.51+00:00,visible,2026,vitalia 150 gps,vitalia 150 gps,30/03/2026,MX3412-italika-vitalia-150-gps,Italika,Vitalia 150 GPS,2026.0,available,32999.0,0.0,32999.0,vitalia 150 gps,both
3,Italika,motocicleta italika dm300 dorado con gps,https://www.italika.mx/motocicleta-italika-dm300-dorado-con-gps-34006890/p,"42,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:06.181+00:00,visible,2026,dm300 gps,dm 300 gps,30/03/2026,MX3401-italika-dm-300-gps,Italika,DM 300 GPS,2026.0,available,43999.0,0.0,43999.0,dm 300 gps,both
4,Italika,motocicleta deportiva italika vort x 250 roja con negro,https://www.italika.mx/motocicleta-deportiva-italika-vort-x-250-roja-con-negro-34005120/p,"49,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,changed,2026-03-26T20:26:06.119+00:00,visible,2026,vort x 250,vort x 250,30/03/2026,MX2802-italika-vort-x-250,Italika,Vort X 250,2026.0,no_stock,50999.0,0.0,50999.0,vort x 250,both
5,Italika,motocicleta de trabajo italika ft125 ts roja con negro,https://www.italika.mx/motocicleta-de-trabajo-italika-ft125-ts-roja-con-negro-34005164/p,"18,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:05.777+00:00,visible,2026,ft125 ts,ft 125 ts,30/03/2026,MX1309-italika-ft-125-ts,Italika,FT 125 TS,2026.0,available,19999.0,0.0,19999.0,ft 125 ts,both
6,Italika,motocicleta de trabajo italika ft150 ts negro con verde,https://www.italika.mx/motocicleta-de-trabajo-italika-ft150-ts-negro-con-verde-34005332/p,"21,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:15.556+00:00,visible,2026,ft150 ts,ft 150 ts,30/03/2026,MX1311-italika-ft-150-ts,Italika,FT 150 TS,2025.0,available,19999.0,0.0,19999.0,ft 150 ts,both
7,Italika,motocicleta de trabajo italika ft150 ts negro con verde,https://www.italika.mx/motocicleta-de-trabajo-italika-ft150-ts-negro-con-verde-34005332/p,"21,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:15.556+00:00,visible,2026,ft150 ts,ft 150 ts,30/03/2026,MX1311-italika-ft-150-ts,Italika,FT 150 TS,2026.0,no_stock,22999.0,0.0,22999.0,ft 150 ts,both
8,Italika,motoneta italika d150 blanco con azul,https://www.italika.mx/motoneta-italika-d150-blanco-con-azul-34006669/p,"22,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:16.38+00:00,visible,2026,d150,d150,30/03/2026,MX1304-italika-d150,Italika,D150,2026.0,no_stock,23999.0,0.0,23999.0,d150,both
9,Italika,motoneta italika d150 blanco con azul,https://www.italika.mx/motoneta-italika-d150-blanco-con-azul-34006669/p,"22,999",contado,MXN,2026-03-31T20:28:05.167446+00:00,same,2026-03-26T20:26:16.38+00:00,visible,2026,d150,d150,30/03/2026,MX1304-italika-d150,Italika,D150,2025.0,available,19999.0,0.0,19999.0,d150,both


## 8. Construir comparacion de precios

In [14]:
import pandas as pd

GALGO_FEE = 1000

_FINAL_COLUMNS = [
    "captured_at", "previous_scrape_at",
    "code", "brand", "model", "model_name", "year", "year_scraped", "status",
    "price_scraped", "price_scraped_with_galgo_fee", "price_base", "discount_amount",
    "price_net", "price_diff",
    "price_type", "currency", "change_status", "visibility",
    "url_scraped", "marketplace_url",
]


def build_price_comparison(
    df_scraped: pd.DataFrame,
    df_inventory: pd.DataFrame,
    country: str,
) -> pd.DataFrame:
    """Construye el DataFrame final de comparación de precios para Italika.

    Une el resultado del scraping con el inventario, aplica el fee de Galgo
    (+1000 MXN, exclusivo de Italika) y calcula la diferencia de precio.

    Args:
        df_scraped: DataFrame con columnas model_mapped y los campos del scrape.
        df_inventory: DataFrame del inventario filtrado por marca, con columna model_lower.
        country: Código de país (ej. 'MX'), usado para construir la URL de marketplace.

    Returns:
        DataFrame con columnas definidas en _FINAL_COLUMNS.
    """
    df = df_scraped.copy()
    df_inv = df_inventory.copy()
    df_inv["model_lower"] = df_inv["model"].str.lower()

    df_merged = pd.merge(
        df,
        df_inv,
        left_on="model_mapped",
        right_on="model_lower",
        how="left",
        indicator=True,
    )

    df_merged.rename(
        columns={"price": "price_scraped", "url": "url_scraped"},
        inplace=True,
    )

    df_merged["marketplace_url"] = df_merged["code"].apply(
        lambda code: f"https://www.galgo.com/{country.lower()}/motos/{code}"
    )

    price_clean = (
        df_merged["price_scraped"]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    df_merged["price_scraped_with_galgo_fee"] = (
        pd.to_numeric(price_clean, errors="coerce") + GALGO_FEE
    )

    df_merged["price_diff"] = (
        df_merged["price_scraped_with_galgo_fee"] - df_merged["price_net"]
    )

    existing_cols = [c for c in _FINAL_COLUMNS if c in df_merged.columns]
    return df_merged[existing_cols].copy()


df_final = build_price_comparison(df_scraped, df_inventory, COUNTRY)
df_final

,captured_at,previous_scrape_at,code,brand,model,model_name,year,year_scraped,status,price_scraped,price_scraped_with_galgo_fee,price_base,discount_amount,price_net,price_diff,price_type,currency,change_status,visibility,url_scraped,marketplace_url
0,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:05.848+00:00,MX1314-italika-rc200,Italika,RC200,motocicleta chopper italika rc200 gris,2025.0,2026,available,"27,999",28999.0,27999.0,0.0,27999.0,1000.0,contado,MXN,same,visible,https://www.italika.mx/motocicleta-chopper-italika-rc200-gris-34005722/p,https://www.galgo.com/mx/motos/MX1314-italika-rc200
1,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:05.848+00:00,MX1314-italika-rc200,Italika,RC200,motocicleta chopper italika rc200 gris,2026.0,2026,no_stock,"27,999",28999.0,28999.0,0.0,28999.0,0.0,contado,MXN,same,visible,https://www.italika.mx/motocicleta-chopper-italika-rc200-gris-34005722/p,https://www.galgo.com/mx/motos/MX1314-italika-rc200
2,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:05.51+00:00,MX3412-italika-vitalia-150-gps,Italika,Vitalia 150 GPS,motoneta italika vitalia 150 beige con gps,2026.0,2026,available,"31,999",32999.0,32999.0,0.0,32999.0,0.0,contado,MXN,same,visible,https://www.italika.mx/motoneta-italika-vitalia-150-beige-con-gps-34006773/p,https://www.galgo.com/mx/motos/MX3412-italika-vitalia-150-gps
3,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:06.181+00:00,MX3401-italika-dm-300-gps,Italika,DM 300 GPS,motocicleta italika dm300 dorado con gps,2026.0,2026,available,"42,999",43999.0,43999.0,0.0,43999.0,0.0,contado,MXN,same,visible,https://www.italika.mx/motocicleta-italika-dm300-dorado-con-gps-34006890/p,https://www.galgo.com/mx/motos/MX3401-italika-dm-300-gps
4,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:06.119+00:00,MX2802-italika-vort-x-250,Italika,Vort X 250,motocicleta deportiva italika vort x 250 roja con negro,2026.0,2026,no_stock,"49,999",50999.0,50999.0,0.0,50999.0,0.0,contado,MXN,changed,visible,https://www.italika.mx/motocicleta-deportiva-italika-vort-x-250-roja-con-negro-34005120/p,https://www.galgo.com/mx/motos/MX2802-italika-vort-x-250
5,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:05.777+00:00,MX1309-italika-ft-125-ts,Italika,FT 125 TS,motocicleta de trabajo italika ft125 ts roja con negro,2026.0,2026,available,"18,999",19999.0,19999.0,0.0,19999.0,0.0,contado,MXN,same,visible,https://www.italika.mx/motocicleta-de-trabajo-italika-ft125-ts-roja-con-negro-34005164/p,https://www.galgo.com/mx/motos/MX1309-italika-ft-125-ts
6,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:15.556+00:00,MX1311-italika-ft-150-ts,Italika,FT 150 TS,motocicleta de trabajo italika ft150 ts negro con verde,2025.0,2026,available,"21,999",22999.0,19999.0,0.0,19999.0,3000.0,contado,MXN,same,visible,https://www.italika.mx/motocicleta-de-trabajo-italika-ft150-ts-negro-con-verde-34005332/p,https://www.galgo.com/mx/motos/MX1311-italika-ft-150-ts
7,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:15.556+00:00,MX1311-italika-ft-150-ts,Italika,FT 150 TS,motocicleta de trabajo italika ft150 ts negro con verde,2026.0,2026,no_stock,"21,999",22999.0,22999.0,0.0,22999.0,0.0,contado,MXN,same,visible,https://www.italika.mx/motocicleta-de-trabajo-italika-ft150-ts-negro-con-verde-34005332/p,https://www.galgo.com/mx/motos/MX1311-italika-ft-150-ts
8,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:16.38+00:00,MX1304-italika-d150,Italika,D150,motoneta italika d150 blanco con azul,2026.0,2026,no_stock,"22,999",23999.0,23999.0,0.0,23999.0,0.0,contado,MXN,same,visible,https://www.italika.mx/motoneta-italika-d150-blanco-con-azul-34006669/p,https://www.galgo.com/mx/motos/MX1304-italika-d150
9,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:16.38+00:00,MX1304-italika-d150,Italika,D150,motoneta italika d150 blanco con azul,2025.0,2026,available,"22,999",23999.0,19999.0,0.0,19999.0,4000.0,contado,MXN,same,visible,https://www.italika.mx/motoneta-italika-d150-blanco-con-azul-34006669/p,https://www.galgo.com/mx/motos/MX1304-italika

## 9. Inspeccion

In [15]:
# Modelos scrapeados sin match en inventario
sin_match = df_merged[df_merged["_merge"] == "left_only"][["model_name", "model_name_clean", "model_mapped"]].drop_duplicates()
print(f"Sin match en inventario: {len(sin_match)}")
sin_match

Sin match en inventario: 3


,model_name,model_name_clean,model_mapped
14,motocicleta de trabajo italika at125 grafito,at125,at125
43,motocicleta de trabajo italika ft200 gts gris con gps,ft200 gts gps,ft200 gts gps
54,motocicleta urbana italika 280z con gps gris,280z gps,280z gps


In [16]:
# Filas sin codigo de inventario
df_final[df_final["code"].isna()]

,captured_at,previous_scrape_at,code,brand,model,model_name,year,year_scraped,status,price_scraped,price_scraped_with_galgo_fee,price_base,discount_amount,price_net,price_diff,price_type,currency,change_status,visibility,url_scraped,marketplace_url
14,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:26:24.801+00:00,NaN,NaN,NaN,motocicleta de trabajo italika at125 grafito,NaN,2026,NaN,"19,999",20999.0,NaN,NaN,NaN,NaN,contado,MXN,same,visible,https://www.italika.mx/motocicleta-de-trabajo-italika-at125-grafito-34006712/p,https://www.galgo.com/mx/motos/nan
43,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:27:03.739+00:00,NaN,NaN,NaN,motocicleta de trabajo italika ft200 gts gris con gps,NaN,2026,NaN,"29,999",30999.0,NaN,NaN,NaN,NaN,contado,MXN,same,visible,https://www.italika.mx/motocicleta-de-trabajo-italika-ft200-gts-gris-con-gps-34007052/p,https://www.galgo.com/mx/motos/nan
54,2026-03-31T20:28:05.167446+00:00,2026-03-26T20:27:22.115+00:00,NaN,NaN,NaN,motocicleta urbana italika 280z con gps gris,NaN,2026,NaN,"45,999",46999.0,NaN,NaN,NaN,NaN,contado,MXN,same,visible,https://www.italika.mx/motocicleta-urbana-italika-280z-con-gps-gris-34006887/p,https://www.galgo.com/mx/motos/nan


## 10. Exportar

In [17]:
output_file = f"31032026-{BRAND_NAME}_precios.csv"
df_final.to_csv(output_file, index=False)
print(f"Archivo exportado: {output_file} — {len(df_final)} filas")

Archivo exportado: 31032026-Italika_precios.csv — 56 filas


---
## Anexo: correr sobre catalogo completo

In [18]:
# # Descomentar para correr sobre todas las URLs validas del catalogo
# rows_full = run_price_tracking(
#     urls=valid_urls,
#     brand_name=BRAND_NAME,
# )
# df_full = pd.DataFrame(rows_full)
# print(f"Modelos procesados: {len(df_full)}")
# print(df_full["change_status"].value_counts().to_dict())
# df_full.head()